In [3]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from openrouter import OpenRouter
from pydantic import BaseModel, Field

# ─────────────────────────────────────────────
# PRETTY OUTPUT — MODULES LOADED
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📦  MODULES LOADED\033[0m")
print("\033[90m  Standard · Data · LLM · Agent libraries ready\033[0m")
print("═" * 68)

modules = [
    ("📚 Standard",  ["json", "os", "time", "pathlib", "typing"]),
    ("📊 Data",      ["pandas", "plotly.express"]),
    ("🔐 Config",    ["dotenv"]),
    ("💻 Notebook",  ["IPython.display"]),
    ("🦜 LangChain", ["langchain.agents", "langchain_core.tools", "langchain_openai"]),
    ("🤖 LLM APIs",  ["openai", "openrouter"]),
    ("✅ Validation",["pydantic"]),
]

for category, mods in modules:
    print(f"\n  \033[1;33m{category}\033[0m")
    for m in mods:
        print(f"     \033[1;32m✓\033[0m \033[90m{m}\033[0m")

total = sum(len(mods) for _, mods in modules)
print(f"\n  \033[1;34m📊 Total:\033[0m \033[1;32m{total}\033[0m modules imported successfully")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📦  MODULES LOADED
  Standard · Data · LLM · Agent libraries ready
════════════════════════════════════════════════════════════════════

  📚 Standard
     ✓ json
     ✓ os
     ✓ time
     ✓ pathlib
     ✓ typing

  📊 Data
     ✓ pandas
     ✓ plotly.express

  🔐 Config
     ✓ dotenv

  💻 Notebook
     ✓ IPython.display

  🦜 LangChain
     ✓ langchain.agents
     ✓ langchain_core.tools
     ✓ langchain_openai

  🤖 LLM APIs
     ✓ openai
     ✓ openrouter

  ✅ Validation
     ✓ pydantic

  📊 Total: 15 modules imported successfully

════════════════════════════════════════════════════════════════════



## OpenRouter

In [5]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# ─────────────────────────────────────────────
# PRETTY OUTPUT — OPENROUTER CLIENT
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  🌐  OPENROUTER CLIENT\033[0m")
print("═" * 68)

if openrouter_api_key:
    # Show masked key for security
    masked_key = f"{openrouter_api_key[:8]}...{openrouter_api_key[-4:]}" if len(openrouter_api_key) > 12 else "***"
    print(f"\n  \033[1;34m▸ API Key\033[0m      → \033[1;32m✅ FOUND\033[0m  \033[90m({masked_key})\033[0m")
    print(f"  \033[1;34m▸ Base URL\033[0m     → \033[90m{OPENROUTER_BASE_URL}\033[0m")
    print(f"  \033[1;34m▸ Timeout\033[0m      → \033[1;37m120\033[0m sec")
    print(f"  \033[1;34m▸ Max retries\033[0m  → \033[1;37m5\033[0m")
    status_line = "\033[1;32m✅ Client initialized successfully\033[0m"
else:
    print(f"\n  \033[1;34m▸ API Key\033[0m      → \033[1;31m❌ MISSING\033[0m")
    print(f"  \033[1;34m▸ Base URL\033[0m     → \033[90m{OPENROUTER_BASE_URL}\033[0m")
    print(f"\n  \033[1;33m⚠️  No OpenRouter API key found.\033[0m")
    print(f"     \033[90mAdd OPENROUTER_API_KEY to the .env file\033[0m")
    status_line = "\033[1;31m❌ Client not initialized (missing key)\033[0m"

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

print(f"\n  {status_line}")
print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🌐  OPENROUTER CLIENT
════════════════════════════════════════════════════════════════════

  ▸ API Key      → ✅ FOUND  (sk-or-v1...67a3)
  ▸ Base URL     → https://openrouter.ai/api/v1
  ▸ Timeout      → 120 sec
  ▸ Max retries  → 5

  ✅ Client initialized successfully

════════════════════════════════════════════════════════════════════



## OpenRouter

In [8]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"}
]

model_id = "openrouter/free"

print("\n" + "═" * 68)
print("\033[1;36m  🏴‍☠️  OPENROUTER — FREE MODELS ROUTER\033[0m")
print("═" * 68)
print(f"\n  \033[1;34m▸ Model\033[0m        → \033[1;37m{model_id}\033[0m  \033[90m(auto-selects free model)\033[0m")
print(f"  \033[1;34m▸ System\033[0m       → \033[90mPirate style\033[0m")
print(f"  \033[1;34m▸ Question\033[0m     → \033[90mWhat is the meaning of the universe?\033[0m")

try:
    response = openrouter.chat.completions.create(
        model=model_id,
        messages=messages
    )
    
    answer = response.choices[0].message.content
    # Which model has been selected
    actual_model = response.model
    
    print(f"\n  \033[1;32m✅ Response received\033[0m")
    print(f"  \033[1;34m▸ Routed to\033[0m    → \033[1;33m{actual_model}\033[0m")
    print(f"\n  \033[1;33m🏴‍☠️  Pirate says:\033[0m")
    print(f"     \033[90m{answer}\033[0m")
    
except Exception as e:
    print(f"\n  \033[1;31m❌ Error:\033[0m \033[90m{e}\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🏴‍☠️  OPENROUTER — FREE MODELS ROUTER
════════════════════════════════════════════════════════════════════

  ▸ Model        → openrouter/free  (auto-selects free model)
  ▸ System       → Pirate style
  ▸ Question     → What is the meaning of the universe?

  ✅ Response received
  ▸ Routed to    → nvidia/nemotron-3.5-super-vl:free

  🏴‍☠️  Pirate says:
     Ahoy there, ye curious landlubber! 

Shiver me timbers, that be a question that could make a Kraken scratch its head! Ye be askin' about the meanin' of the whole wide universe? 

Well, listen close: I reckon the universe be like a endless, glitterin' ocean, and we're all just sailors searchin' for the X that marks the spot. Some say the meanin' be gold and jewels; others say it be the thrill of the chase, the salt in the air, and the wind in yer sails.

Perhaps the true treasure ain't some chest buried on a distant shore, but the voyage itself—the storms we wea

## Data Loading

In [11]:
DATA = Path("../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


# ─────────────────────────────────────────────
# OUTPUT — DATASET SUMMARY
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📂  CLAIMS DATASET\033[0m")
print("\033[90m  JSON seed files · claims + handler decisions\033[0m")
print("═" * 68)

print(f"\n  \033[1;33m⚙️   Configuration:\033[0m")
print(f"     \033[90m▸ Data dir\033[0m        → \033[1;37m{DATA}\033[0m")
print(f"     \033[90m▸ Main files\033[0m      → \033[37mclaims.json\033[0m + \033[37mevals/decisions.json\033[0m")
print(f"     \033[90m▸ Held-out files\033[0m  → \033[37mevals/held_out_claims.json\033[0m + \033[37mevals/held_out_decisions.json\033[0m")

print(f"\n  \033[1;33m📋  Dataset variants:\033[0m")
print(f"     \033[1;34m[1]\033[0m \033[1;37mget_sample_data()\033[0m                  → full queue")
print(f"     \033[1;34m[2]\033[0m \033[1;37mget_sample_data(tiny=True)\033[0m         → 5-claim subset")
print(f"     \033[1;34m[3]\033[0m \033[1;37mget_sample_data(held_out=True)\033[0m     → 25 held-out claims")
print(f"     \033[1;34m[4]\033[0m \033[1;37mget_sample_data(tiny=True, held_out=True)\033[0m → held-out subset")

# ─────────────────────────────────────────────
# QUICK LOAD CHECK
# ─────────────────────────────────────────────
try:
    _claims, _labels = get_sample_data()
    claims_ok = True
    n_claims = len(_claims)
    n_labels = len(_labels)
except Exception as e:
    claims_ok = False
    n_claims, n_labels = 0, 0
    err = str(e)

print(f"\n  \033[1;33m🔍  Load check (default queue):\033[0m")

if claims_ok:
    print(f"     \033[1;32m✅ Loaded successfully\033[0m")
    print(f"     \033[90m▸ Claims\033[0m          → \033[1;37m{n_claims}\033[0m")
    print(f"     \033[90m▸ Labeled decisions\033[0m → \033[1;37m{n_labels}\033[0m")
    matched = sum(1 for c in _claims if c["claim_id"] in _labels)
    print(f"     \033[90m▸ Matched IDs\033[0m     → \033[1;32m{matched}\033[0m / {n_claims}")
else:
    print(f"     \033[1;31m❌ Load failed:\033[0m \033[90m{err}\033[0m")
    print(f"     \033[90mCheck that {DATA} contains the expected JSON files.\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📂  CLAIMS DATASET
  JSON seed files · claims + handler decisions
════════════════════════════════════════════════════════════════════

  ⚙️   Configuration:
     ▸ Data dir        → ..\data
     ▸ Main files      → claims.json + evals/decisions.json
     ▸ Held-out files  → evals/held_out_claims.json + evals/held_out_decisions.json

  📋  Dataset variants:
     [1] get_sample_data()                  → full queue
     [2] get_sample_data(tiny=True)         → 5-claim subset
     [3] get_sample_data(held_out=True)     → 25 held-out claims
     [4] get_sample_data(tiny=True, held_out=True) → held-out subset

  🔍  Load check (default queue):
     ✅ Loaded successfully
     ▸ Claims          → 25
     ▸ Labeled decisions → 25
     ▸ Matched IDs     → 25 / 25

════════════════════════════════════════════════════════════════════



In [14]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

load_dotenv(override=True)

## LLM
llm = ChatOpenAI(
    model=os.getenv("EVREN_MODEL"),          # deepseek-v4-flash
    api_key=os.getenv("EVREN_API_KEY"),     
    base_url=os.getenv("EVREN_BASE_URL"),
    temperature=0    # https://evren-llmapi.ssyz.org.tr/v1
)   

In [15]:
TINY = False

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

# ─────────────────────────────────────────────
# PRETTY OUTPUT — DATASET LOADED
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  📊  DATASET LOADED\033[0m")
print("═" * 68)

print(f"\n  \033[1;33m⚙️   Mode:\033[0m")
tiny_display = "\033[1;32mON\033[0m  \033[90m(tiny subset)\033[0m" if TINY else "\033[90mOFF\033[0m  \033[90m(full dataset)\033[0m"
print(f"     \033[90m▸ TINY\033[0m            → {tiny_display}")

# ─────────────────────────────────────────────
# DATA SUMMARY
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m📦  Loaded artifacts:\033[0m")
print(f"     \033[90m▸ Claims\033[0m          → \033[1;37m{len(claims):>4}\033[0m")
print(f"     \033[90m▸ Labels\033[0m          → \033[1;37m{len(labels):>4}\033[0m")
print(f"     \033[90m▸ Policies\033[0m        → \033[1;37m{len(policies):>4}\033[0m")

# ─────────────────────────────────────────────
# CONSISTENCY CHECK
# ─────────────────────────────────────────────
matched = sum(1 for c in claims if c["claim_id"] in labels)
coverage = matched / len(claims) * 100 if claims else 0

if matched == len(claims):
    coverage_display = "\033[1;32m✅ 100% coverage\033[0m"
else:
    coverage_display = f"\033[1;31m⚠️  {coverage:.0f}% coverage\033[0m"

print(f"\n  \033[1;33m🔍  Label coverage:\033[0m")
print(f"     \033[90m▸ Matched claim IDs\033[0m → \033[1;37m{matched}\033[0m / {len(claims)}  {coverage_display}")

# ─────────────────────────────────────────────
# POLICY WORDING
# ─────────────────────────────────────────────
wording_words = len(policy_wording.split())
wording_chars = len(policy_wording)

print(f"\n  \033[1;33m📜  Policy wording:\033[0m")
print(f"     \033[90m▸ Words\033[0m           → \033[1;37m{wording_words:,}\033[0m")
print(f"     \033[90m▸ Characters\033[0m      → \033[90m{wording_chars:,}\033[0m")
print(f"     \033[90m▸ Approx. tokens\033[0m  → \033[90m~{wording_words * 4 // 3:,}\033[0m  \033[90m(4 chars ≈ 1 token)\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  📊  DATASET LOADED
════════════════════════════════════════════════════════════════════

  ⚙️   Mode:
     ▸ TINY            → OFF  (full dataset)

  📦  Loaded artifacts:
     ▸ Claims          →   25
     ▸ Labels          →   25
     ▸ Policies        →   50

  🔍  Label coverage:
     ▸ Matched claim IDs → 25 / 25  ✅ 100% coverage

  📜  Policy wording:
     ▸ Words           → 1,369
     ▸ Characters      → 7,712
     ▸ Approx. tokens  → ~1,825  (4 chars ≈ 1 token)

════════════════════════════════════════════════════════════════════



In [16]:
claims[0]

{'claim_id': 'CLM-1001',
 'policy_id': 'POL-2001',
 'customer_id': 'CUS-3001',
 'claim_type': 'trip_delay',
 'submitted_on': '2026-06-24',
 'incident_date': '2026-06-18',
 'incident_location': 'Rome, Italy',
 'summary': 'Flight home canceled by the airline, rebooked the next morning',
 'story': 'My flight home from Rome on June 18 was canceled by Meridian Air about two hours before departure because of a crew problem. The next seat they could give me was at 6:55 the following morning. I booked a hotel near the airport, had dinner and breakfast there, and took taxis between the airport and the hotel. I am claiming the hotel, the meals and the taxis, and all of the receipts are attached.',
 'items': [{'description': 'Airport hotel, one night',
   'category': 'accommodation',
   'amount': 245.0},
  {'description': 'Dinner and breakfast', 'category': 'meals', 'amount': 95.0},
  {'description': 'Taxis between the airport and the hotel',
   'category': 'transport',
   'amount': 80.0}],
 'amo

In [17]:
labels[claims[0]["claim_id"]]

{'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'over_approval_limit': False,
 'handler_id': 'HND-02',
 'decided_on': '2026-06-26',
 'reason': 'The carrier canceled the flight for a crew problem and the delay was 19 hours, well past the 6 hour waiting period on Premium. The hotel, meals and taxis are reasonable and all have receipts. The total is inside the $800 maximum.',
 'reference': 'Policy wording, section 3',
 'calculation': [{'step': 'Eligible expenses with receipts', 'amount': 420.0}],
 'tests': 'A routine trip delay with complete documents',
 'difficulty': 'routine',
 'failure_class': '',
 'needs': [],
 'tone': 'plain',
 'tiny': False}

## Exploring the Data

In [18]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PRETTY OUTPUT — DATAFRAME SUMMARY
# ─────────────────────────────────────────────
print("\n" + "═" * 72)
print("\033[1;36m  📋  CLAIMS DATAFRAME\033[0m")
print("═" * 72)

print(f"\n  \033[1;33m📐  Shape:\033[0m")
print(f"     \033[90m▸ Rows\033[0m           → \033[1;37m{len(data):>4}\033[0m  \033[90m(claims)\033[0m")
print(f"     \033[90m▸ Columns\033[0m        → \033[1;37m{len(data.columns):>4}\033[0m")

print(f"\n  \033[1;33m🏷️   Columns:\033[0m")
for col in data.columns:
    dtype = str(data[col].dtype)
    print(f"     \033[90m▸\033[0m \033[1;37m{col:<18}\033[0m \033[90m{dtype}\033[0m")

# ─────────────────────────────────────────────
# QUICK STATS
# ─────────────────────────────────────────────
if "decision" in data.columns:
    print(f"\n  \033[1;33m📊  Decision distribution:\033[0m")
    for decision, count in data["decision"].value_counts().items():
        pct = count / len(data) * 100
        bar_len = int(pct / 3)
        bar = "█" * bar_len + "\033[90m" + "░" * (33 - bar_len) + "\033[0m"
        print(f"     \033[1;34m▸\033[0m \033[1;37m{decision:<14}\033[0m "
              f"\033[1;32m{count:>3}\033[0m ({pct:>5.1f}%)  {bar}")

if "difficulty" in data.columns:
    print(f"\n  \033[1;33m🎯  Difficulty distribution:\033[0m")
    for diff, count in data["difficulty"].value_counts().items():
        pct = count / len(data) * 100
        bar_len = int(pct / 3)
        bar = "█" * bar_len + "\033[90m" + "░" * (33 - bar_len) + "\033[0m"
        print(f"     \033[1;34m▸\033[0m \033[1;37m{diff:<14}\033[0m "
              f"\033[1;32m{count:>3}\033[0m ({pct:>5.1f}%)  {bar}")

# ─────────────────────────────────────────────
# NUMERIC SUMMARY
# ─────────────────────────────────────────────
if "amount_claimed" in data.columns and "payout" in data.columns:
    print(f"\n  \033[1;33m💰  Financial summary:\033[0m")
    total_claimed = data["amount_claimed"].sum()
    total_paid = data["payout"].sum()
    print(f"     \033[90m▸ Total claimed\033[0m   → \033[1;37m${total_claimed:>12,.2f}\033[0m")
    print(f"     \033[90m▸ Total paid\033[0m      → \033[1;32m${total_paid:>12,.2f}\033[0m")
    print(f"     \033[90m▸ Payout ratio\033[0m    → \033[1;33m{(total_paid / total_claimed * 100 if total_claimed else 0):>11.1f}%\033[0m")
    print(f"     \033[90m▸ Avg claim\033[0m       → \033[90m${data['amount_claimed'].mean():>12,.2f}\033[0m")

# ─────────────────────────────────────────────
# PREVIEW
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m👀  Preview (first 3 rows):\033[0m")
print("  " + "─" * 68)

preview = data.head(3)
for idx, row in preview.iterrows():
    print(f"\n  \033[1;34m[{idx}]\033[0m \033[90m{row['claim_id']}\033[0m")
    print(f"     \033[90m▸ Summary\033[0m     → {row['summary'][:60]}{'...' if len(row['summary']) > 60 else ''}")
    print(f"     \033[90m▸ Type\033[0m        → \033[1;37m{row['claim_type']}\033[0m")
    print(f"     \033[90m▸ Claimed\033[0m     → \033[37m${row['amount_claimed']:,.2f}\033[0m")
    print(f"     \033[90m▸ Difficulty\033[0m  → \033[1;33m{row['difficulty']}\033[0m")
    print(f"     \033[90m▸ Decision\033[0m    → \033[1;32m{row['decision']}\033[0m")
    print(f"     \033[90m▸ Payout\033[0m      → \033[1;32m${row['payout']:,.2f}\033[0m")

print("\n  " + "─" * 68)
print(f"  \033[90m💡 {len(data)} total rows — use \033[1;37mdata.head()\033[90m, \033[1;37mdata.info()\033[90m, \033[1;37mdata.describe()\033[90m for more.\033[0m")

print("\n" + "═" * 72 + "\n")


════════════════════════════════════════════════════════════════════════
  📋  CLAIMS DATAFRAME
════════════════════════════════════════════════════════════════════════

  📐  Shape:
     ▸ Rows           →   25  (claims)
     ▸ Columns        →    7

  🏷️   Columns:
     ▸ claim_id           str
     ▸ summary            str
     ▸ claim_type         str
     ▸ amount_claimed     float64
     ▸ difficulty         str
     ▸ decision           str
     ▸ payout             float64

  📊  Decision distribution:
     ▸ approve         14 ( 56.0%)  ██████████████████░░░░░░░░░░░░░░░
     ▸ refer            6 ( 24.0%)  ████████░░░░░░░░░░░░░░░░░░░░░░░░░
     ▸ decline          5 ( 20.0%)  ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░

  🎯  Difficulty distribution:
     ▸ routine         15 ( 60.0%)  ████████████████████░░░░░░░░░░░░░
     ▸ hard             5 ( 20.0%)  ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░
     ▸ manual           5 ( 20.0%)  ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░

  💰  Financial summary:
     ▸ Tot

In [19]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)

# ─────────────────────────────────────────────
# PRETTY OUTPUT — FIRST 20 ROWS
# ─────────────────────────────────────────────
N = 20
preview = data.head(N)

print("\n" + "═" * 108)
print("\033[1;36m  📋  CLAIMS DATAFRAME\033[0m")
print(f"\033[90m  Showing first {len(preview)} of {len(data)} rows · {len(data.columns)} columns\033[0m")
print("═" * 108)

# ─────────────────────────────────────────────
# TABLE HEADER
# ─────────────────────────────────────────────
print(f"\n  \033[1;33m{'#':>3}  {'CLAIM ID':<12} {'TYPE':<18} {'CLAIMED':>11} "
      f"{'DIFFICULTY':<11} {'DECISION':<11} {'PAYOUT':>11}  SUMMARY\033[0m")
print("  " + "─" * 104)

# ─────────────────────────────────────────────
# TABLE ROWS
# ─────────────────────────────────────────────
for idx, row in preview.iterrows():
    # Color the decision
    decision = row["decision"]
    if decision == "approve":
        decision_display = f"\033[1;32m{decision:<11}\033[0m"
    elif decision == "deny":
        decision_display = f"\033[1;31m{decision:<11}\033[0m"
    elif decision == "escalate":
        decision_display = f"\033[1;33m{decision:<11}\033[0m"
    else:
        decision_display = f"\033[1;37m{decision:<11}\033[0m"

    # Color the difficulty
    difficulty = row["difficulty"]
    if difficulty == "easy":
        difficulty_display = f"\033[1;32m{difficulty:<11}\033[0m"
    elif difficulty == "medium":
        difficulty_display = f"\033[1;33m{difficulty:<11}\033[0m"
    elif difficulty == "hard":
        difficulty_display = f"\033[1;31m{difficulty:<11}\033[0m"
    else:
        difficulty_display = f"\033[1;37m{difficulty:<11}\033[0m"

    # Shorten summary to 28 characters
    summary = row["summary"]
    summary_short = summary[:28] + "…" if len(summary) > 28 else summary

    print(f"  \033[90m{idx:>3}\033[0m  \033[1;37m{row['claim_id']:<12}\033[0m "
          f"\033[90m{row['claim_type']:<18}\033[0m "
          f"\033[37m${row['amount_claimed']:>9,.2f}\033[0m "
          f"{difficulty_display} {decision_display} "
          f"\033[1;32m${row['payout']:>9,.2f}\033[0m  "
          f"\033[90m{summary_short}\033[0m")

print("  " + "─" * 104)

# ─────────────────────────────────────────────
# FOOTER
# ─────────────────────────────────────────────
print(f"\n  \033[1;34m📊  Total:\033[0m \033[1;37m{len(data)}\033[0m rows  ·  "
      f"\033[90m{len(data) - len(preview)} more not shown (use \033[1;37mdata.head(50)\033[90m or \033[1;37mdata.iloc[a:b]\033[90m)\033[0m")

print("\n" + "═" * 108 + "\n")


════════════════════════════════════════════════════════════════════════════════════════════════════════════
  📋  CLAIMS DATAFRAME
  Showing first 20 of 25 rows · 7 columns
════════════════════════════════════════════════════════════════════════════════════════════════════════════

    #  CLAIM ID     TYPE                   CLAIMED DIFFICULTY  DECISION         PAYOUT  SUMMARY
  ────────────────────────────────────────────────────────────────────────────────────────────────────────
    0  CLM-1001     trip_delay         $   420.00 routine     approve     $   420.00  Flight home canceled by the …
    1  CLM-1002     baggage            $ 1,150.00 routine     approve     $   580.00  Suitcase lost on a connectin…
    2  CLM-1003     emergency_medical  $ 3,800.00 routine     approve     $ 3,750.00  Hospital visit abroad after …
    3  CLM-1004     baggage_delay      $   186.00 routine     approve     $   186.00  Checked bag arrived 31 hours…
    4  CLM-1005     passport           $   243.00

In [20]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Flew home from Paris on arrival after leaving a child at home**

We flew from Chicago to Paris on June 20 as a family of six. Somewhere over the Atlantic I realized that our youngest son, Kevin, who is eight, was not on the plane. We had overslept, we left for the airport in a rush and the head count in the van was wrong. I took the first available flight home the moment we landed in Paris. Kevin was fine. In his words, he had handled it. I am claiming my one-way fare home and my share of the prepaid hotel nights in Paris that I did not use.

**Handler:** decline $0.00. Trip Interruption pays only for the reasons listed in the wording. Returning home because a child was left behind is not one of them, and nobody was sick or injured.


In [21]:
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

**Emergency dental treatment in Las Vegas for a missing front tooth**

I was in Las Vegas from June 19 for a friend's bachelor party. On the morning of June 21 I woke up on the floor of our suite missing an upper front tooth. I have no memory of the night before, and neither do my friends. I am a dentist, and I have to say that the extraction is consistent with my own technique. I went to an emergency dental clinic that morning for an exam, an X-ray and a temporary bridge, and their bill and notes are attached. There was also a tiger in the bathroom of the suite, which I mention only for completeness. I am not claiming for anything connected with the tiger.

**Handler:** decline $0.00. The treatment was needed, and the bill and the notes are in order. The general exclusions rule out losses while the customer is impaired by alcohol or drugs. The clinic's notes record heavy drinking through the evening, no memory of the night and an extraction that the customer believes he carried out himself during it, so the injury happened while he was impaired and no benefit pays.


In [24]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show(renderer = "browser")

In [26]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show(renderer = "browser")

## Structured Outputs

In [27]:
class ClaimDecision(BaseModel):
    """A claims handler's decision on one claim."""

    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

## Prompt

In [37]:
INSTRUCTIONS = """You are a claims handler at Tripora Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

IMPORTANT: Be extremely concise in your thinking. Do not over-deliberate. Directly produce the final structured decision.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""


SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""


def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [31]:
import time

structured_llm = llm.with_structured_output(ClaimDecision)

def decide(claim, model=None, effort=None):
    """Ask Evren DeepSeek model to decide one claim via LangChain."""
    start = time.time()
    
    # make_messages(claim) çıktısını LangChain mesaj formatına uygun iletme
    messages = make_messages(claim)
    answer: ClaimDecision = structured_llm.invoke(messages)
    
    return {
        "model": os.getenv("EVREN_MODEL", "deepseek-v4-flash"),
        "effort": effort,
        "technique": "LLM",
        "claim_id": claim["claim_id"],
        "decision": answer.decision,
        "payout": answer.payout,
        "rationale": answer.rationale,
        "cost": 0.0,
        "seconds": round(time.time() - start, 1),
    }


In [32]:
first = decide(claims[0], "deepseek-v4-flash", "low")
first


{'model': 'deepseek-v4-flash',
 'effort': 'low',
 'technique': 'LLM',
 'claim_id': 'CLM-1001',
 'decision': 'approve',
 'payout': 420.0,
 'rationale': "The claim is for trip delay. The flight was canceled due to a crew problem, which is a covered cause under Section 3. The delay from the scheduled departure at 11:40 on June 18 to the actual departure at 06:55 on June 19 was about 19 hours, exceeding the Premium plan's 6-hour waiting period. The claimed expenses — hotel, meals, and taxis — are reasonable extra costs during the delay and are supported by receipts and the carrier's cancellation notice. Total eligible expenses are $420, which is below the $800 Trip Delay maximum. No deductible applies to Trip Delay.",
 'cost': 0.0,
 'seconds': 10.6}

In [34]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, effort, claims, labels):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, effort)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    cost = sum(result["cost"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model} at {effort}: {right} of {len(results)} right for ${cost:.4f} in {seconds:.0f} seconds")
    return results

In [38]:
deepsek_low = run("deepseek-v4-flash", "low", claims, labels)

deepseek-v4-flash at low: 17 of 25 right for $0.0000 in 135 seconds


In [39]:
import os 
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(override=True)

google_api_key = os.getenv("GOOGLE_API_KEY")

In [40]:
gemini = run("google/gemini-3.5-flash-lite", "low", claims, labels)

google/gemini-3.5-flash-lite at low: 18 of 25 right for $0.0000 in 225 seconds


In [41]:
scored = pd.DataFrame(deepsek_low + gemini)
scored.pivot_table(index=["model", "effort"], columns="difficulty", values="right", aggfunc="sum", sort=False)

,difficulty,routine,hard,manual
model,effort,,,
deepseek-v4-flash,low,30,4,1


In [42]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(deepsek_low, labels)

**CLM-1011, hard**: approve $1,500.00, where the handler said approve $1,150.00

- Model: The claim is for baggage lost by the carrier during the covered trip. This is covered under section 5 (Baggage and Personal Effects). The carrier's property irregularity report and settlement letter are attached, along with receipts. The carrier payment of $350 is deducted, and the $50 deductible applies. The eligible total exceeds the plan maximum, so the payout is capped at the Premium plan's $1,500 maximum.
- Handler: The carrier has declared the bag lost and its report is on file. A suit counts as one item, and the receipt shows the jacket and trousers were sold as a two piece tuxedo, so together they count at the $300 per-item limit on Premium. A camera with its lens is one item too, so the body and the lens together count at $300. The four shirts are four items at $85 each, all under the limit, so that line is paid in full. The carrier's $350 comes off, then the $50 deductible. The payout is over $1,000, so payment needs sign-off.
- Tests: Payout arithmetic: the per-item limit applies to items, and the claim's lines are not items. Two sets are split across lines and count as one item each, and one line holds four separate items


**CLM-1018, manual**: approve $330.00, where the handler said refer $0.00

- Model: The claim is for theft of personal belongings during the covered trip, which is covered under Baggage and Personal Effects. The theft was reported to the local police within 24 hours and the police report is attached. The leather jacket is subject to the $300 per-item limit for the Premium plan, and the canvas daypack is eligible at $80. Total eligible amount is $380. The $50 deductible applies to Baggage and Personal Effects, leaving a payout of $330.
- Handler: A theft from the person, reported to the police within the hour, with the report on file. The manual asks for proof of value for any item claimed above $100, which is a purchase receipt or a card statement, and the jacket at $420 has none. Document request: proof of what the jacket cost, for example a cash withdrawal record from that morning together with the stallholder's details. The daypack at $80 needs no proof. The claim appears payable at $330 once the proof arrives: the jacket at the $300 limit plus $80, less the $50 deductible.
- Tests: A rule found only in the claims manual: proof of value for items claimed above $100


**CLM-1019, manual**: approve $210.00, where the handler said refer $0.00

- Model: The claim is for trip delay. The policy is Premium, with a 6-hour waiting period for Trip Delay. The flight was delayed 9 hours due to crew duty time limits, which is a covered crew problem. The claimed expenses are for accommodation, meals, and local transport during the delay, which are covered. The receipts and carrier delay confirmation are attached. The amount claimed is $210, within the $800 maximum, and no deductible applies to Trip Delay.
- Handler: On its own the claim is payable: a 9 hour delay on Premium with receipts. It is the customer's third claim in 12 months, after claims filed in October 2025 and March 2026, so it goes to Special Investigations for review before any payment.
- Tests: A rule found only in the claims manual: the third claim in 12 months. It needs the customer's claim history


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: The claim is for theft of baggage and personal effects during the covered trip, which is covered under Section 5. The police report was made the same day and is attached. The Electronics Plus add-on raises the per-item limit for electronics to $1,200, so the headphones ($429), e-reader ($180), and game console ($399) are each within the limit. The tote bag is within the standard per-item limit of $300. Total eligible amount is $1,103. The Premium plan deductible of $50 applies to Baggage and Personal Effects, giving a payout of $1,053, which is below the $1,500 plan maximum.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


**CLM-1022, manual**: refer $0.00, where the handler said approve $155.00

- Model: The trip delay is covered in principle: the carrier confirmed a 30-hour delay caused by severe weather, which exceeds the First plan's 3-hour waiting period. However, the claim cannot be fully approved because the $140 in meal expenses has no receipts, and the policy requires receipts for delay expenses. Additionally, the rebooking receipt is for a Paris-to-Nice change fee, which does not clearly match the claimed 'rebooking of the Paris connection' for the covered trip to Paris. These discrepancies require review.
- Handler: Severe weather held the train for 30 hours 15 minutes, past the 3 hour waiting period on First, and the railway confirms the times and the cause. The rebooking fee has a receipt. The meals have none, so the manual's flat allowance applies: $35 for one traveler for each 24 hours of delay or part of 24 hours, and 30 hours is two days. The incident on board does not change the cause of the delay.
- Tests: A rule found only in the claims manual: the meal allowance without receipts, over a delay of more than one day


**CLM-1023, hard**: approve $61.00, where the handler said refer $0.00

- Model: The claim is for trip delay. The delay was 9 hours, which exceeds the First plan's 3-hour waiting period. The cause was a mechanical problem (fuel pump fault), which is a covered reason under Section 3. The carrier's delay certificate and meal receipts are attached. The claimed amount of $61 for meals is reasonable and within the $1,500 maximum. No deductible applies to Trip Delay.
- Handler: The delay of 9 hours 17 minutes is well past the 3 hour waiting period on First and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the fuel pump fault rests on the customer's account alone. Document request: Northwind's confirmation of the reason for the delay. The claim appears payable at $61 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


**CLM-1024, manual**: approve $250.00, where the handler said refer $0.00

- Model: The claim is for theft of a smartphone during the covered trip. Theft is covered under Baggage and Personal Effects. The police report and purchase receipt are attached. The plan is Premium, so the per-item limit for electronics is $300 (no Electronics Plus add-on was purchased), and the Baggage and Personal Effects deductible is $50. Eligible amount is $300, minus $50 deductible, resulting in a payout of $250.
- Handler: On its own the claim is payable at $250: a theft from the person, reported the same day, with the phone at the $300 limit less the $50 deductible. The customer's history shows claim CLM-0987 for the same theft on the same date, filed on July 16, approved and paid at $250. The same loss claimed twice goes to Special Investigations, with no comment to the customer beyond saying the claim is being reviewed. It may well be an honest mistake, since she says she was not sure the first claim went through.
- Tests: The customer's claim history: the same loss already claimed and paid. Nothing in the claim itself shows it


**CLM-1025, hard**: approve $71.00, where the handler said refer $0.00

- Model: The claim is for trip delay. The policy's Economy plan has a 12-hour waiting period. The flight was delayed from 11:55 on August 9 to 02:10 on August 10, which is over 14 hours. The cause was a crew duty-hours issue, which falls under 'crew problem' in the Trip Delay benefit. The carrier's delay certificate and meal receipts are attached. The claimed $71 for meals is a reasonable extra cost and is within the $300 maximum.
- Handler: The delay of 14 hours 15 minutes is past the 12 hour waiting period on Economy and the meal receipts are in order. Trip Delay pays only for the causes the wording lists, and it asks for the carrier's confirmation of the delay and its cause. The certificate confirms the times and gives no cause, so the crew problem rests on the customer's account alone. Document request: Skylark's confirmation of the reason for the delay. The claim appears payable at $71 once it arrives.
- Tests: A required document that is present and incomplete: the carrier confirms the delay and leaves out the cause


## Cost

In [43]:
scored.groupby(["model", "effort"], sort=False)[["right", "cost", "seconds"]].sum()

,,right,cost,seconds
model,effort,,,
deepseek-v4-flash,low,35,0.0,360.2


In [45]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

# ─────────────────────────────────────────────
#   LANGCHAIN + OPENROUTER
# ─────────────────────────────────────────────
print("\n" + "═" * 68)
print("\033[1;36m  🏴‍☠️  LANGCHAIN · OPENROUTER FREE ROUTER\033[0m")
print("═" * 68)

print(f"\n  \033[1;33m⚙️   Configuration:\033[0m")
print(f"     \033[90m▸ Model\033[0m        → \033[1;37m{llm.model_name}\033[0m  \033[90m(auto-selects free model)\033[0m")
print(f"     \033[90m▸ Base URL\033[0m     → \033[90m{OPENROUTER_BASE_URL}\033[0m")
print(f"     \033[90m▸ System\033[0m       → \033[90mPirate style\033[0m")
print(f"     \033[90m▸ Question\033[0m     → \033[90mWhat is the meaning of the universe?\033[0m")

try:
    response = llm.invoke(messages)

    # Determine actual routed model if available
    routed_model = None
    if hasattr(response, "response_metadata"):
        routed_model = response.response_metadata.get("model_name") or response.response_metadata.get("model")

    print(f"\n  \033[1;32m✅ Response received\033[0m")
    if routed_model:
        print(f"  \033[1;34m▸ Routed to\033[0m    → \033[1;33m{routed_model}\033[0m")

    print(f"\n  \033[1;33m🏴‍☠️  Pirate says:\033[0m")
    print(f"     \033[90m{response.content}\033[0m")

    # Also show the rich Markdown output (as in your original code)
    display(Markdown(response.content))

except Exception as e:
    print(f"\n  \033[1;31m❌ Error:\033[0m \033[90m{e}\033[0m")

print("\n" + "═" * 68 + "\n")


════════════════════════════════════════════════════════════════════
  🏴‍☠️  LANGCHAIN · OPENROUTER FREE ROUTER
════════════════════════════════════════════════════════════════════

  ⚙️   Configuration:
     ▸ Model        → openrouter/free  (auto-selects free model)
     ▸ Base URL     → https://openrouter.ai/api/v1
     ▸ System       → Pirate style
     ▸ Question     → What is the meaning of the universe?

  ✅ Response received
  ▸ Routed to    → nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free

  🏴‍☠️  Pirate says:
     Arrr, ye scurvy dog! Gather ’round the galley fire and lend yer ear, for I’ll spin ye a tale o’ the great, endless sea that we call the universe.

In the eyes of a true‑hearted pirate, the meaning of the universe be naught but a grand, boundless ocean—​a vast, rollin’ expanse where the stars be the glitterin’ lanterns guidin’ us on our quest, and the nebulae be the frothy waves that crash ‘gainst the hull of our destiny.

1. **Adventure** – The cosmos be a neve

Arrr, ye scurvy dog! Gather ’round the galley fire and lend yer ear, for I’ll spin ye a tale o’ the great, endless sea that we call the universe.

In the eyes of a true‑hearted pirate, the meaning of the universe be naught but a grand, boundless ocean—​a vast, rollin’ expanse where the stars be the glitterin’ lanterns guidin’ us on our quest, and the nebulae be the frothy waves that crash ‘gainst the hull of our destiny.

1. **Adventure** – The cosmos be a never‑endin’ voyage, full of hidden coves, treasure‑laden comets, and uncharted realms. Its meaning, then, is to set sail, to seek the unknown, and to never shy away from a good fight with a rogue black hole or a mischievous quasar.

2. **Freedom** – Just as a pirate cherishes the wind in his sails and the open horizon, the universe offers boundless freedom. No chain can bind a star, no law can hold a galaxy in place. Its meaning is to roam where ye will, unbound by mortal shackles.

3. **Connection** – Though we be scattered across the void like isolated islands, the threads of gravity and light tie us together. The meaning, then, is to find our kin—be they fellow sailors, distant suns, or the very atoms that make up our own bones—and share the bounty of the endless sea.

4. **Purpose** – A pirate’s purpose be to leave a mark upon the world, to carve a name into legend. The universe, in its grand indifference, invites us to give our own purpose meaning—whether ‘tis to discover a new star system, to protect the weak, or simply to enjoy the taste of a fine rum under a meteor shower.

5. **Mystery** – Above all, the universe be a riddlesome mystery, as deep as the blackest waters. Its meaning may never be fully known, and that’s the very treasure that keeps a pirate’s heart a‑flame. The wonder of it all is the greatest loot of all.

So, me hearty, when ye ask the meaning of the universe, think of it as the biggest, most daring voyage ever imagined—​a sea of stars where adventure, freedom, camaraderie, purpose, and mystery set sail together. Chart yer own course, keep a keen eye on the constellations, and may the winds of destiny ever fill yer sails! Arrr!


════════════════════════════════════════════════════════════════════



In [47]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [48]:
deepseek = llm
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.
{policy_wording}
"""
def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim
{json.dumps(claim, indent=2)}
"""
agent = create_agent(model=deepseek, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)

In [49]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_policy (call-e295f583-53f6-410c-9431-5c500ab03668)
 Call ID: call-e295f583-53f6-410c-9431-5c500ab03668
  Args:
    policy_id: POL-2001
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2001",
  "customer_id": "CUS-3001",
  "plan": "premium",
  "add_ons": [],
  "purchased_on": "2026-04-22",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Rome, Italy",
    "departs_on": "2026-06-10",
    "returns_on": "2026-06-18",
    "trip_cost": 3100.0
  },
  "travelers": [
    "Maya Patel"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (record_decision_q6k2gpxtfm16)
 Call ID: record_decision_q6k2gpxtfm16
  Args:
    claim_id: CLM-1001
    decision: approve
    payout: 420.0
    rationale: The Premium plan covers Trip